<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/GPT_repl_Karpathy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [14]:
chars=sorted(set(text))
vocab_size= len(chars)

In [61]:
stoi={j:i for i,j in enumerate(chars)}
itos={i:j for i,j in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[s] for s in l])

In [47]:
data = torch.tensor(encode(text), dtype=torch.long)
n=int(len(data)*0.9)
train_data=data[:n]
val_data=data[n:]

In [58]:
block_size=8

train_data[:block_size+1]


KeyError: tensor(18)

In [63]:
x=train_data[:block_size]
y=train_data[1:block_size+1]
for i in range(block_size):
  context=x[:i+1]
  target=y[i]
  print(f"when context is {context} the target is {target}")

when context is tensor([18]) the target is 47
when context is tensor([18, 47]) the target is 56
when context is tensor([18, 47, 56]) the target is 57
when context is tensor([18, 47, 56, 57]) the target is 58
when context is tensor([18, 47, 56, 57, 58]) the target is 1
when context is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when context is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when context is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [154]:
torch.manual_seed(1337)

batch_size=4
block_size=8


def get_batch(split="Train"):
  data= train_data if split =="Train" else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size]for i in ix])
  y=torch.stack([data[i+1:i+block_size+1]for i in ix])
  return x, y
xb,yb = get_batch()
for b in range(batch_size):
  for i in range(block_size):
    context=xb[b,:i+1]
    target=yb[b,i]


 context is tensor([24]), target is 43
 context is tensor([24, 43]), target is 58
 context is tensor([24, 43, 58]), target is 5
 context is tensor([24, 43, 58,  5]), target is 57
 context is tensor([24, 43, 58,  5, 57]), target is 1
 context is tensor([24, 43, 58,  5, 57,  1]), target is 46
 context is tensor([24, 43, 58,  5, 57,  1, 46]), target is 43
 context is tensor([24, 43, 58,  5, 57,  1, 46, 43]), target is 39
 context is tensor([44]), target is 53
 context is tensor([44, 53]), target is 56
 context is tensor([44, 53, 56]), target is 1
 context is tensor([44, 53, 56,  1]), target is 58
 context is tensor([44, 53, 56,  1, 58]), target is 46
 context is tensor([44, 53, 56,  1, 58, 46]), target is 39
 context is tensor([44, 53, 56,  1, 58, 46, 39]), target is 58
 context is tensor([44, 53, 56,  1, 58, 46, 39, 58]), target is 1
 context is tensor([52]), target is 58
 context is tensor([52, 58]), target is 1
 context is tensor([52, 58,  1]), target is 58
 context is tensor([52, 58, 

In [220]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [221]:
class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table= nn.Embedding(vocab_size, vocab_size)
  def forward(self,idx,targets):
    logits = self.token_embedding_table(idx)

    B,T,C = logits.shape
    logits=logits.view(-1,C)

    loss = F.cross_entropy(logits, targets.view(-1))
    return logits, loss

In [222]:
m= BigramLanguageModel(vocab_size)
out, loss=m(xb,yb)
print(loss)

tensor(4.8786, grad_fn=<NllLossBackward0>)


In [231]:
torch.log(torch.tensor(1/65))

tensor(-4.1744)

In [212]:
xb).shape

torch.Size([4, 8])